<a href="https://colab.research.google.com/github/Mehtabwho/CSE-Lab-Courses/blob/main/8th_sem%20/DM_LAB/Search_engine/Job_search_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# SECTION 1: Install Libraries
!pip install -q gradio nltk pandas numpy

In [2]:
# SECTION 2: Imports
import re
import math
import time
import html
import collections
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import gradio as gr

# Download required NLTK data components
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

True

In [4]:
# SECTION 3: Load Dataset
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

def load_datasets():
    try:
        df_bd = pd.read_csv('/content/drive/MyDrive/DM_LAB/bdjobs.csv', encoding='latin1')
    except Exception:
        df_bd = pd.DataFrame()

    try:
        df_cj = pd.read_csv('/content/drive/MyDrive/DM_LAB/careerjet.csv')
    except Exception:
        df_cj = pd.DataFrame()

    try:
        df_sk = pd.read_csv('/content/drive/MyDrive/DM_LAB/skill.csv', encoding='latin1')
    except Exception:
        df_sk = pd.DataFrame()

    return df_bd, df_cj, df_sk

df_bd, df_cj, df_sk = load_datasets()

Mounted at /content/drive


In [5]:
# SECTION 4: Data Cleaning
processed_list = []

# Clean and Map BDJobs Matrix
if not df_bd.empty:
    for _, row in df_bd.iterrows():
        title = str(row.get('Job name', '')).strip()
        if not title or '???' in title or title == 'nan':
            continue
        company = str(row.get('Company Name', 'Not Specified')).strip()
        location = str(row.get('Location', 'Bangladesh')).strip()
        job_type = str(row.get('Experience ', 'Not Specified')).strip()
        edu = str(row.get('Education Qualification', '')).strip()
        link = str(row.get('Description Link', '#')).strip()

        if company == 'nan': company = 'Not Specified'
        if location == 'nan': location = 'Bangladesh'
        if job_type == 'nan': job_type = 'Not Specified'
        if edu == 'nan': edu = ''

        content = f"{title} {company} {location} {edu}"
        processed_list.append({
            'title': title, 'company': company, 'location': location,
            'job_type': job_type, 'content': content, 'link': link, 'source': 'BDJobs'
        })

# Clean and Map Careerjet Matrix
if not df_cj.empty:
    for _, row in df_cj.iterrows():
        title = str(row.get('job', '')).strip()
        if not title or title == 'nan':
            continue
        company = 'Not Specified'
        location = str(row.get('location', 'Bangladesh')).strip()
        badge = str(row.get('badge', ''))
        badge2 = str(row.get('badge 2', ''))

        job_types = []
        if badge and badge != 'nan': job_types.append(badge)
        if badge2 and badge2 != 'nan': job_types.append(badge2)
        job_type = " / ".join(job_types) if job_types else "Not Specified"

        desc = str(row.get('desc', '')).strip()
        link = str(row.get('job href', '#')).strip()
        if desc == 'nan': desc = ''
        if location == 'nan': location = 'Bangladesh'

        content = f"{title} {location} {desc}"
        processed_list.append({
            'title': title, 'company': company, 'location': location,
            'job_type': job_type, 'content': content, 'link': link, 'source': 'Careerjet'
        })

# Clean and Map Skill.jobs Matrix
if not df_sk.empty:
    for _, row in df_sk.iterrows():
        title = str(row.get('Job Title', '')).strip()
        if not title or title == 'nan':
            continue
        company = str(row.get('Company name', 'Not Specified')).strip()
        location = str(row.get('Location', 'Bangladesh')).strip()
        job_type = str(row.get('Job Type', 'Full Time')).strip()
        link = str(row.get('Job description Link', '#')).strip()

        if company == 'nan': company = 'Not Specified'
        if location == 'nan': location = 'Bangladesh'
        if job_type == 'nan': job_type = 'Full Time'

        content = f"{title} {company} {location}"
        processed_list.append({
            'title': title, 'company': company, 'location': location,
            'job_type': job_type, 'content': content, 'link': link, 'source': 'Skill.jobs'
        })

df_jobs = pd.DataFrame(processed_list)
if not df_jobs.empty:
    df_jobs = df_jobs.dropna(subset=['title', 'content']).reset_index(drop=True)
else:
    df_jobs = pd.DataFrame(columns=['title', 'company', 'location', 'job_type', 'content', 'link', 'source'])
print(f"Total structured jobs loaded into core engine: {len(df_jobs)}")

Total structured jobs loaded into core engine: 7524


In [6]:
# SECTION 5: Text Preprocessing
STOP_WORDS = set(stopwords.words('english'))
STEMMER = PorterStemmer()

def preprocess_text(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = text.split()
    return [STEMMER.stem(token) for token in tokens if token not in STOP_WORDS and len(token) > 1]

In [7]:
# SECTION 6: Build Inverted Index
inverted_index = collections.defaultdict(dict)

for doc_id, row in df_jobs.iterrows():
    tokens = preprocess_text(row['content'])
    for token in tokens:
        if doc_id in inverted_index[token]:
            inverted_index[token][doc_id] += 1
        else:
            inverted_index[token][doc_id] = 1
print("Inverted index map populated successfully.")

Inverted index map populated successfully.


In [8]:
# SECTION 7: TF-IDF Computation
N = len(df_jobs)
idf = {}

for term, postings in inverted_index.items():
    df_t = len(postings)
    idf[term] = math.log((N / df_t) + 1) if df_t > 0 else 0.0

print("Global structural TF-IDF score computation matrix established.")

Global structural TF-IDF score computation matrix established.


In [9]:
# SECTION 8: Boolean Query Processing (AND / OR)
def process_boolean_query(query_text, mode="OR"):
    query_upper = query_text.upper()
    if " AND " in query_upper:
        parts = re.split(r'\s+AND\s+', query_text, flags=re.IGNORECASE)
        mode = "AND"
        terms = []
        for p in parts: terms.extend(preprocess_text(p))
    elif " OR " in query_upper:
        parts = re.split(r'\s+OR\s+', query_text, flags=re.IGNORECASE)
        mode = "OR"
        terms = []
        for p in parts: terms.extend(preprocess_text(p))
    else:
        terms = preprocess_text(query_text)

    if not terms:
        return set(), []

    doc_sets = []
    for term in terms:
        if term in inverted_index:
            doc_sets.append(set(inverted_index[term].keys()))
        else:
            if mode == "AND":
                doc_sets.append(set())

    if not doc_sets:
        return set(), terms

    if mode == "AND":
        return set.intersection(*doc_sets), terms
    else:
        return set.union(*doc_sets), terms

In [10]:
# SECTION 9: Ranking Engine
def rank_documents(candidate_docs, query_terms):
    scores = collections.defaultdict(float)
    for doc_id in candidate_docs:
        score = 0.0
        for term in query_terms:
            if term in inverted_index and doc_id in inverted_index[term]:
                tf = inverted_index[term][doc_id]
                score += tf * idf.get(term, 0.0)
        scores[doc_id] = score
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [11]:
# SECTION 11: Snippet Generation
def generate_snippet(content, query_terms):
    if not content or not isinstance(content, str):
        return "No descriptive telemetry available."
    words = content.split()
    words_lower = [w.lower() for w in words]
    match_word_idx = 0
    found = False

    for i, w in enumerate(words_lower):
        for term in query_terms:
            if term in w or STEMMER.stem(w) == term:
                match_word_idx = i
                found = True
                break
        if found: break

    start_idx = max(0, match_word_idx - 10)
    end_idx = min(len(words), match_word_idx + 15)
    snippet = " ".join(words[start_idx:end_idx])

    if start_idx > 0: snippet = "... " + snippet
    if end_idx < len(words): snippet = snippet + " ..."

    for term in query_terms:
        try:
            escaped_term = re.escape(term)
            snippet = re.sub(r'(\b\w*' + escaped_term + r'\w*\b)', r'<mark class="highlight">\1</mark>', snippet, flags=re.IGNORECASE)
        except Exception:
            pass
    return snippet

In [12]:
# SECTION 10: Search Function
def search_engine(query, mode):
    if not query or not query.strip():
        return "<div style='text-align: center; color: rgba(240,248,255,0.4); padding: 40px;'>Please enter query parameters to scan structural matrix.</div>"

    candidate_docs, query_terms = process_boolean_query(query, mode)
    if not candidate_docs:
        return f"<div style='padding: 20px; text-align: center;'><h3 style='color: #f0f8ff;'>No log segments located for: <b>{html.escape(query)}</b></h3></div>"

    start_time = time.time()
    ranked_results = rank_documents(candidate_docs, query_terms)
    elapsed_time = time.time() - start_time

    html_output = f"<div style='font-family: sans-serif; max-width: 750px; margin: 0 auto;'><p style='color: #38bdf8; font-size: 14px;'>Optimized {len(ranked_results)} database nodes mapped in ({elapsed_time:.4f} seconds)</p>"

    for doc_id, score in ranked_results[:10]:
        row = df_jobs.iloc[doc_id]
        snippet = generate_snippet(row['content'], query_terms)

        html_output += f"""
        <div class="result-card" style="margin-bottom: 24px; background: rgba(10,30,60,0.35); padding: 20px; border-radius: 12px; border: 1px solid rgba(56,189,248,0.15); backdrop-filter: blur(8px);">
            <div style="font-size: 12px; margin-bottom: 8px; display: flex; align-items: center; gap: 8px;">
                <span style="background: linear-gradient(135deg, #00f2fe 0%, #4facfe 100%); padding: 2px 8px; border-radius: 4px; font-weight: bold; color: #020c1b; text-transform: uppercase; font-size:10px;">{row['source']}</span>
                <span style="color: rgba(240,248,255,0.4); font-family: monospace;">{row['link'][:65]}...</span>
            </div>
            <h3 style="margin: 0 0 6px 0; font-size: 19px;"><a href="{row['link']}" target="_blank" style="color: #38bdf8; text-decoration: none;">{row['title']}</a></h3>
            <div style="font-size: 13px; color: #cbd5e1; margin-bottom: 10px; display: flex; gap: 14px;">
                <span>🏢 {row['company']}</span><span>📍 {row['location']}</span>
                <span style="border: 0.5px solid rgba(56,189,248,0.3); padding: 0 6px; border-radius: 4px; color:#38bdf8;">{row['job_type']}</span>
                <span style="color: #81c784; font-weight: bold; margin-left: auto;">Score: {score:.3f}</span>
            </div>
            <p style="margin: 0; font-size: 14px; color: #94a3b8; line-height: 1.6;">{snippet}</p>
        </div>
        """
    return html_output + "</div>"

In [18]:
# SECTION 12: Modern Gradio Blocks UI

# Encapsulating structural layouts and neon ambient glows
custom_cyber_css = """
footer {display: none !important;}
.gradio-container {background: #020c1b !important; border:none !important; max-width: 950px !important; padding: 20px !important;}

/* Hero Banner Layout Framework */
.hero-wrapper {
    position: relative; width: 100%; min-height: 280px;
    background: radial-gradient(circle at center, #0a192f 0%, #020c1b 100%) !important;
    display: flex; flex-direction: column; align-items: center; justify-content: center;
    overflow: hidden; padding: 30px 20px; box-sizing: border-box; border-radius: 16px;
    border: 1px solid rgba(56,189,248,0.15) !important; margin-bottom: 25px;
}
.ambient-orb { position: absolute; border-radius: 50%; filter: blur(90px); opacity: 0.12; pointer-events: none; }
.orb-1 { width: 350px; height: 350px; background: #00f2fe; top: -10%; left: 5%; }
.orb-2 { width: 400px; height: 400px; background: #4facfe; bottom: -20%; right: 5%; }

/* Typography */
.logo-box {
    display: flex; align-items: center; gap: 10px; background: linear-gradient(135deg, #00f2fe 0%, #4facfe 100%);
    padding: 6px 18px; border-radius: 9px; box-shadow: 0 0 25px rgba(0, 242, 254, 0.3); margin-bottom: 15px;
}
.logo-text { font-weight: 700 !important; font-size: 18px !important; color: #020c1b !important; letter-spacing: 1.5px !important; }
.main-title { font-size: 38px !important; font-weight: 700 !important; color: #ffffff !important; margin: 0 0 6px 0 !important; }
.subtitle { font-size: 14px !important; color: #38bdf8 !important; opacity: 0.9 !important; margin: 0 !important; }

/* Structural Overrides for Native Inputs to achieve Glassmorphism */
#search-input-field textarea, #search-input-field input {
    background: rgba(10, 30, 60, 0.75) !important;
    border: 1px solid rgba(56, 189, 248, 0.4) !important;
    color: #ffffff !important; font-size: 16px !important;
    border-radius: 24px !important; padding: 12px 20px !important;
    box-shadow: 0 0 15px rgba(0,0,0,0.5) !important;
}
#search-input-field textarea:focus, #search-input-field input:focus {
    border-color: rgba(0, 242, 254, 0.8) !important;
    box-shadow: 0 0 20px rgba(0, 242, 254, 0.25) !important;
}

/* Cyber Matrix Action Button Trigger */
.cyber-btn {
    background: linear-gradient(135deg, #00f2fe 0%, #4facfe 100%) !important;
    border: none !important; border-radius: 24px !important; color: #020c1b !important;
    font-weight: bold !important; font-size: 15px !important; cursor: pointer !important;
    box-shadow: 0 0 15px rgba(0, 242, 254, 0.3) !important; transition: all 0.2s;
}
.cyber-btn:hover { transform: scale(1.02); box-shadow: 0 0 25px rgba(0, 242, 254, 0.5) !important; }

/* Logic Selector Box Custom Matrix Styles */
.radio-matrix { background: rgba(10, 30, 60, 0.4) !important; border: 1px solid rgba(56,189,248,0.15) !important; border-radius: 12px !important; padding: 10px !important; }

/* Global Highlight Utility */
.highlight { background-color: rgba(0, 242, 254, 0.25) !important; border-bottom: 1.5px solid #00f2fe; color: #ffffff !important; font-weight: bold; }
"""

with gr.Blocks(css=custom_cyber_css, title="Job Zen Search Matrix") as demo:

    # Hero Segment Frame Render
    gr.HTML("""
    <div class="hero-wrapper">
        <div class="ambient-orb orb-1"></div><div class="ambient-orb orb-2"></div>
        <div class="logo-box"><div class="logo-text"><h1>JOB ZEN</h1></div></div>
        <h1 class="main-title">Find Your Next Role</h1>
        <p class="subtitle">Search thousands of opportunities instantly with intelligent matching.</p>
    </div>
    """)

    # Controlled Input System Area
    with gr.Row():
        query_input = gr.Textbox(
            placeholder="Search jobs by title, skills, company, location, or keywords...",
            show_label=False,
            scale=8,
            max_lines=1,
            container=False,
            elem_id="search-input-field"
        )
        action_trigger = gr.Button("Search", scale=2, elem_classes=["cyber-btn"])

    with gr.Row(variant="panel", elem_classes=["radio-matrix"]):
        with gr.Column(scale=3):
            gr.Markdown("<span style='color:#38bdf8; font-weight:bold; font-size:13px;'>CONTROL PARAMETERS :</span>")
        with gr.Column(scale=7):
            logic_mode = gr.Radio(
                choices=["OR", "AND"],
                value="OR",
                label="Evaluation Match Logic Rules",
                container=False
            )

    # Output Cluster Stream Area
    search_results_container = gr.HTML(
        value="<div style='text-align: center; color: rgba(56,189,248,0.5); padding: 50px; font-family: sans-serif; font-size:14px; border: 1.5px dashed rgba(56,189,248,0.15); border-radius:12px; margin-top:20px;'></div>"
    )

    # Dynamic Engine Processing Triggers
    query_input.submit(fn=search_engine, inputs=[query_input, logic_mode], outputs=search_results_container)
    action_trigger.click(fn=search_engine, inputs=[query_input, logic_mode], outputs=search_results_container)

demo.launch(debug=True)

/tmp/ipykernel_2135/3631636000.py:58: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_cyber_css, title="Job Zen Search Matrix") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://52884783bdc95f0bd1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://52884783bdc95f0bd1.gradio.live
